✅ Cell 1 — Setup (GPU check + installs)

In [ ]:
!which nvidia-smi && nvidia-smi || echo "No GPU detected (CPU runtime)"
!pip -q install tqdm


In [ ]:
import os, time, random, shutil
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


✅ Cell 2 — Download + extract CUB-200-2011

In [3]:
!rm -f /content/CUB_200_2011.tgz
!rm -rf /content/CUB_200_2011

!wget -q --show-progress https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz
!tar -xvf /content/CUB_200_2011.tgz -C /content >/dev/null

!ls /content/CUB_200_2011 | head


CUB_200_2011.tgz    100%[===================>]   1.07G  18.8MB/s    in 62s     
attributes
bounding_boxes.txt
classes.txt
image_class_labels.txt
images
images.txt
parts
README
train_test_split.txt


✅ Cell 3 — Build train/test folders (ImageFolder format)

In [4]:
from pathlib import Path

CUB_ROOT = Path("/content/CUB_200_2011")
OUT_ROOT = Path("/content/ProtoViT/datasets/cub200")
TRAIN_OUT = OUT_ROOT / "train"
TEST_OUT  = OUT_ROOT / "test"

# clean output
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

# images.txt: <img_id> <relative_path>
images = {}
with open(CUB_ROOT / "images.txt", "r") as f:
    for line in f:
        img_id, rel = line.strip().split()
        images[int(img_id)] = rel

# train_test_split.txt: <img_id> <is_train(1/0)>
split = {}
with open(CUB_ROOT / "train_test_split.txt", "r") as f:
    for line in f:
        img_id, is_train = line.strip().split()
        split[int(img_id)] = int(is_train)

src_images_root = CUB_ROOT / "images"

count_train = 0
count_test = 0

for img_id, rel_path in images.items():
    cls = rel_path.split("/")[0]
    src = src_images_root / rel_path

    if split[img_id] == 1:
        dst_dir = TRAIN_OUT / cls
        count_train += 1
    else:
        dst_dir = TEST_OUT / cls
        count_test += 1

    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / Path(rel_path).name
    shutil.copy2(src, dst)

print("✅ Done!")
print("Train images:", count_train)
print("Test images :", count_test)
print("Output:", OUT_ROOT)


✅ Done!
Train images: 5994
Test images : 5794
Output: /content/ProtoViT/datasets/cub200


✅ Cell 4 — Verify folder structure

In [5]:
!ls /content/ProtoViT/datasets/cub200
!ls /content/ProtoViT/datasets/cub200/train | head


test  train
001.Black_footed_Albatross
002.Laysan_Albatross
003.Sooty_Albatross
004.Groove_billed_Ani
005.Crested_Auklet
006.Least_Auklet
007.Parakeet_Auklet
008.Rhinoceros_Auklet
009.Brewer_Blackbird
010.Red_winged_Blackbird


✅ Cell 5 — DataLoaders

In [6]:
DATA_ROOT = "/content/ProtoViT/datasets/cub200"
TRAIN_DIR = f"{DATA_ROOT}/train"
TEST_DIR  = f"{DATA_ROOT}/test"

assert os.path.isdir(TRAIN_DIR), f"Missing {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR),  f"Missing {TEST_DIR}"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=test_tf)

print("Classes:", len(train_ds.classes))
print("Train samples:", len(train_ds), "| Test samples:", len(test_ds))

BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)


Classes: 200
Train samples: 5994 | Test samples: 5794


✅ Cell 6 — Model + Optimizer

In [7]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)  # faster option

model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)

EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("✅ Model ready. AMP:", use_amp)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 139MB/s]


✅ Model ready. AMP: False


/tmp/ipython-input-4266587411.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


✅ Cell 7 — Train/Eval Functions

In [8]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return loss_sum / total, correct / total

def train_one_epoch(model, loader):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

        pbar.set_postfix(loss=float(loss.item()), acc=correct/total)

    return loss_sum / total, correct / total


✅ Cell 8 — Training Loop (WITH HISTORY LOGGING ✅)

In [ ]:
# ✅ ADDED: history containers (this is what you were missing)
train_losses, val_losses = [], []
train_accs, val_accs = [], []

best_acc = 0.0
SAVE_DIR = Path("/content/resnet_cub200_run")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = SAVE_DIR / "best.pt"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, test_loader)
    scheduler.step()

    # ✅ ADDED: save per-epoch metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    took = time.time() - t0
    lr_now = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch:02d}/{EPOCHS} | lr {lr_now:.6f} | "
          f"train acc {train_acc:.4f} | val acc {val_acc:.4f} | "
          f"loss {train_loss:.4f}/{val_loss:.4f} | {took:.1f}s")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "model": model.state_dict(),
            "classes": train_ds.classes,
            "epoch": epoch,
            "val_acc": val_acc,
        }, BEST_PATH)
        print(f"✅ Saved best: {BEST_PATH} (acc={best_acc:.4f})")

print("✅ Done. Best val acc:", best_acc)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/94 [00:00<?, ?it/s]

/tmp/ipython-input-3544617282.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


✅ Cell 9 — Training Curves (Loss + Accuracy)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accs, label="Train Accuracy")
plt.plot(epochs, val_accs, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training & Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


Predict one random test image (and print labels)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

raw = Image.open(img_path).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(raw)
plt.axis("off")
plt.title(f"True: {true_name}\nPred: {pred_name}")
plt.show()
